# E1 — JVM Heap Saturation Analysis

**Experiment:** E1-jvm-heap  
**Layer:** L1 — JVM image knob (`JAVA_TOOL_OPTIONS` / `-Xmx`)  
**Scenarios:** S0-baseline vs S1-heap-256m  
**Workload:** W2-upload at {low, medium, high} intensity

## Research question
When the JVM heap is undersized (`-Xmx256m`), does the recommender correctly:
1. Detect the diagnostic signal: `gc_pause_rate > 1/s` AND `heap_ratio > 0.9` both sustained
2. Attribute the root cause to **L1** (not L3 or L4)
3. Propose the correct fix: `-Xmx2g`

## Figures produced
- **Fig 1** — Diagnostic time series: heap utilization ratio + GC pause count (S0 vs S1, medium intensity)
- **Fig 2** — p99 latency and error rate comparison by scenario × intensity
- **Fig 3** — Evidence link: heap_ratio_max vs p99 scatter

Run `python experiments/E1-jvm-heap/run.py --intensities medium` first to generate results.

In [ ]:
import json
import csv
import sys
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import yaml
import pandas as pd
from IPython.display import display

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

REPO_ROOT = Path('.').resolve().parent.parent
RESULTS_DIR = REPO_ROOT / 'results'
FIGURES_DIR = Path('.') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

EXPERIMENT_KEY = 'E1-jvm-heap'
SCENARIOS      = ['S0-baseline', 'S1-heap-256m']
INTENSITIES    = ['low', 'medium', 'high']

COLORS = {
    'S0-baseline':  '#2196F3',
    'S1-heap-256m': '#F44336',
}
LABELS = {
    'S0-baseline':  'S0 baseline (Xmx=2g)',
    'S1-heap-256m': 'S1 heap-256m (Xmx=256m)',
}
INTENSITY_POS = {'low': 0, 'medium': 1, 'high': 2}

In [ ]:
def load_locust_stats(run_dir: Path) -> dict:
    stats_file = run_dir / 'locust_stats.csv'
    if not stats_file.exists():
        return {}
    rows = {}
    with open(stats_file) as f:
        for row in csv.DictReader(f):
            name = row.get('Name', '')
            try:
                total = float(row.get('Request Count', 1) or 1)
                rows[name] = {
                    'p50':        float(row.get('50%', 0) or 0),
                    'p95':        float(row.get('95%', 0) or 0),
                    'p99':        float(row.get('99%', 0) or 0),
                    'error_rate': float(row.get('Failure Count', 0) or 0) / max(1, total),
                    'rps':        float(row.get('Requests/s', 0) or 0),
                    'count':      total,
                }
            except (ValueError, TypeError):
                pass
    return rows


def get_prom_series(prom: dict, key: str):
    data = prom.get(key, {})
    if data.get('status') != 'success':
        return [], []
    ts_list, v_list = [], []
    for series in data.get('data', {}).get('result', []):
        for ts, v in series.get('values', []):
            try:
                ts_list.append(float(ts))
                v_list.append(float(v))
            except (ValueError, TypeError):
                pass
    return ts_list, v_list


def load_run(run_id: str) -> dict:
    run_dir = RESULTS_DIR / run_id
    meta = yaml.safe_load((run_dir / 'metadata.yaml').read_text()) \
        if (run_dir / 'metadata.yaml').exists() else {}
    prom = json.loads((run_dir / 'prom_snapshot.json').read_text()) \
        if (run_dir / 'prom_snapshot.json').exists() else {}
    return {'run_id': run_id, 'meta': meta, 'prom': prom,
            'stats': load_locust_stats(run_dir)}

In [ ]:
runs = []
if RESULTS_DIR.exists():
    for run_dir in sorted(RESULTS_DIR.iterdir()):
        mf = run_dir / 'metadata.yaml'
        if not mf.exists():
            continue
        meta = yaml.safe_load(mf.read_text())
        if meta.get('experiment') == EXPERIMENT_KEY:
            runs.append(load_run(run_dir.name))

print(f'Found {len(runs)} run(s) for {EXPERIMENT_KEY}')
for r in runs:
    m = r['meta']
    print(f"  {r['run_id'][:8]}  scenario={m.get('scenario','?')}  "
          f"intensity={m.get('intensity','?')}  workload={m.get('workload','?')}")

if not runs:
    print('\nNo results yet. Generate data with:')
    print('  python experiments/E1-jvm-heap/run.py --intensities medium')

In [ ]:
records = []
for r in runs:
    m, prom, stats = r['meta'], r['prom'], r['stats']
    agg = stats.get('Aggregated', next(iter(stats.values()), {}))

    _, heap_used = get_prom_series(prom, "jvm_memory_used_bytes{area='heap'}")
    _, heap_max  = get_prom_series(prom, "jvm_memory_max_bytes{area='heap'}")
    ratios = [u / mx for u, mx in zip(heap_used, heap_max) if mx > 0]

    _, gc_counts = get_prom_series(prom, 'jvm_gc_pause_seconds_count')

    records.append({
        'scenario':        m.get('scenario', '?'),
        'intensity':       m.get('intensity', '?'),
        'run_id':          r['run_id'][:8],
        'p99_ms':          agg.get('p99', 0),
        'p95_ms':          agg.get('p95', 0),
        'error_rate_%':    round(agg.get('error_rate', 0) * 100, 2),
        'rps':             agg.get('rps', 0),
        'heap_ratio_max':  round(max(ratios), 3) if ratios else 0,
        'heap_ratio_mean': round(float(np.mean(ratios)), 3) if ratios else 0,
        'gc_count_mean':   round(float(np.mean(gc_counts)), 1) if gc_counts else 0,
    })

df = pd.DataFrame(records)
if not df.empty:
    display(df.sort_values(['scenario', 'intensity']))
else:
    print('No data. Run experiments first.')

In [ ]:
# Figure 1 — Diagnostic time series (medium intensity, S0 vs S1)
# This is the 'smoking gun' figure: both signals must fire to trigger diagnosis.

target = {s: None for s in SCENARIOS}
for r in runs:
    sc = r['meta'].get('scenario')
    it = r['meta'].get('intensity')
    if it == 'medium' and sc in target:
        target[sc] = r

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Fig 1 — E1 Diagnostic Signals: Heap Ratio and GC Pause Count (W2-upload, medium intensity)',
    fontsize=12, fontweight='bold', y=1.01
)

for scenario, r in target.items():
    c = COLORS[scenario]
    lbl = LABELS[scenario]

    if r is None:
        for ax in axes:
            ax.text(0.5, 0.5, f'Run for {scenario}\nnot found yet',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=10, color='gray', style='italic')
        continue

    prom = r['prom']

    # Heap ratio
    ts, hu = get_prom_series(prom, "jvm_memory_used_bytes{area='heap'}")
    _,  hm = get_prom_series(prom, "jvm_memory_max_bytes{area='heap'}")
    if ts and hu and hm:
        ratios = [u / mx for u, mx in zip(hu, hm) if mx > 0]
        t_rel  = [(t - ts[0]) / 60 for t in ts[:len(ratios)]]
        axes[0].plot(t_rel, ratios, color=c, label=lbl, linewidth=2)

    # GC pause cumulative count
    ts_gc, gc = get_prom_series(prom, 'jvm_gc_pause_seconds_count')
    if ts_gc and gc:
        t_rel = [(t - ts_gc[0]) / 60 for t in ts_gc]
        axes[1].plot(t_rel, gc, color=c, label=lbl, linewidth=2)

axes[0].axhline(0.90, color='orange', linestyle='--', linewidth=1.2,
                alpha=0.8, label='diagnostic threshold (0.90)')
axes[0].set_title('Heap Utilization Ratio')
axes[0].set_xlabel('Time into run (min)')
axes[0].set_ylabel('heap_used / heap_max')
axes[0].set_ylim(0, 1.15)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axhline(1.0, color='orange', linestyle='--', linewidth=1.2,
                alpha=0.8, label='diagnostic threshold (1/s proxy)')
axes[1].set_title('GC Pause Count (cumulative — rate proxy)')
axes[1].set_xlabel('Time into run (min)')
axes[1].set_ylabel('jvm_gc_pause_seconds_count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'E1-fig1-diagnostic-signals.pdf')
plt.show()
print('Saved: figures/E1-fig1-diagnostic-signals.pdf')

In [ ]:
# Figure 2 — p99 latency and error rate by scenario × intensity (grouped bar chart)

if df.empty:
    print('No data for Figure 2.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Fig 2 — E1 SLO Impact: p99 Latency and Error Rate (S0 vs S1)',
                 fontsize=12, fontweight='bold')

    x     = np.arange(len(INTENSITIES))
    width = 0.35

    for idx, scenario in enumerate(SCENARIOS):
        sub = df[df['scenario'] == scenario].set_index('intensity')
        p99_vals  = [sub.loc[i, 'p99_ms']     if i in sub.index else 0 for i in INTENSITIES]
        err_vals  = [sub.loc[i, 'error_rate_%'] if i in sub.index else 0 for i in INTENSITIES]
        offset = (idx - 0.5) * width
        axes[0].bar(x + offset, p99_vals, width, label=LABELS[scenario],
                    color=COLORS[scenario], alpha=0.85)
        axes[1].bar(x + offset, err_vals,  width, label=LABELS[scenario],
                    color=COLORS[scenario], alpha=0.85)

    axes[0].set_title('p99 Latency (ms)')
    axes[0].set_xlabel('Intensity')
    axes[0].set_ylabel('p99 latency (ms)')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(INTENSITIES)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    axes[1].set_title('Error Rate (%)')
    axes[1].set_xlabel('Intensity')
    axes[1].set_ylabel('Error rate (%)')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(INTENSITIES)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'E1-fig2-slo-impact.pdf')
    plt.show()
    print('Saved: figures/E1-fig2-slo-impact.pdf')

In [ ]:
# Figure 3 — Evidence link: heap_ratio_max vs p99 (scatter)
# Shows the recommender's evidence is grounded in SLO impact.

if df.empty:
    print('No data for Figure 3.')
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    fig.suptitle('Fig 3 — E1 Evidence Link: Heap Utilization vs p99 Latency',
                 fontsize=12, fontweight='bold')

    for scenario in SCENARIOS:
        sub = df[df['scenario'] == scenario]
        ax.scatter(sub['heap_ratio_max'], sub['p99_ms'],
                   color=COLORS[scenario], label=LABELS[scenario],
                   s=80, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(row['intensity'],
                        (row['heap_ratio_max'], row['p99_ms']),
                        textcoords='offset points', xytext=(4, 4), fontsize=9)

    ax.axvline(0.90, color='orange', linestyle='--', linewidth=1.2, alpha=0.8,
               label='diagnostic threshold (0.90)')
    ax.set_xlabel('Max Heap Utilization Ratio')
    ax.set_ylabel('p99 Latency (ms)')
    ax.set_xlim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'E1-fig3-evidence-link.pdf')
    plt.show()
    print('Saved: figures/E1-fig3-evidence-link.pdf')

## Interpretation

**Expected outcome (pre-run prediction):**

| Metric | S0 baseline | S1 heap-256m |
|--------|-------------|---------------|
| heap_ratio_max | < 0.60 | > 0.90 |
| gc_count rate | < 0.5/s | > 1/s |
| p99 latency | baseline | > 5× baseline |
| error rate | < 0.5% | > 1% |

**Diagnostic rule (from diagnose.py):**
```
IF gc_pause_rate > 1/s (sustained 30% of samples)
AND heap_used/heap_max > 0.90 (peak)
THEN scenario = S1-heap-256m, layer = L1, confidence = f(heap_ratio)
```

**Paper claim:** The dual-signal condition prevents false positives.
A GC spike alone (e.g. large object allocation burst) should NOT trigger
the L1 diagnosis if heap_ratio stays healthy. Both signals sustained
together are the fingerprint of heap exhaustion.